# Atelier ML Retail - Exploration et Preparation des Donnees

*GI2 | 2025-2026*

## 0. Imports et Configuration

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.utils import (
    describe_dataframe, detect_outliers_iqr,
    plot_missing_values, plot_correlation_heatmap,
    plot_pca_variance, plot_distributions, plot_boxplots, plot_pca_2d
)
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

## 1. Chargement des Donnees

In [ ]:
df = pd.read_csv('../data/raw/retail_customers.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
describe_dataframe(df)

## 2. Valeurs Manquantes

In [ ]:
plot_missing_values(df)
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## 3. Distributions des Features Numeriques

In [ ]:
num_cols = [
    'Recency', 'Frequency', 'MonetaryTotal', 'MonetaryAvg',
    'Age', 'CustomerTenure', 'SupportTickets', 'Satisfaction',
    'WeekendRatio', 'ReturnRatio', 'UniqueProducts', 'CancelledTrans'
]
plot_distributions(df, num_cols)

## 4. Detection des Outliers (IQR)

In [ ]:
outliers = detect_outliers_iqr(
    df, ['SupportTickets', 'Satisfaction', 'MonetaryTotal', 'TotalQuantity']
)
print(outliers.to_string())
plot_boxplots(df, ['SupportTickets', 'Satisfaction', 'MonetaryTotal', 'ReturnRatio'])

## 5. Analyse du Churn (Variable Cible)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution Churn
vals = df['Churn'].value_counts()
axes[0].bar(['Fidele (0)', 'Churn (1)'], vals.values, color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Distribution du Churn')
for i, v in enumerate(vals.values):
    pct = v / len(df)
    axes[0].text(i, v + 5, f'{v} ({pct:.1%})', ha='center')

# Churn par segment RFM
churn_seg = df.groupby('RFMSegment')['Churn'].mean().sort_values()
churn_seg.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Taux de churn par segment RFM')
axes[1].set_xlabel('Taux de churn')

plt.tight_layout()
plt.show()

## 6. Correlations entre Features

In [ ]:
num_df = df.select_dtypes(include=['number']).drop(columns=['CustomerID'], errors='ignore')
# Heatmap correlation
corr = plot_correlation_heatmap(num_df, threshold=0.85)

In [ ]:
# Paires fortement correlees avec Churn
churn_corr = num_df.corrwith(num_df['Churn']).abs().sort_values(ascending=False)
print('Top 15 features correlees avec Churn :')
print(churn_corr.head(15))

## 7. Analyse en Composantes Principales (ACP)

Objectif : reduire les 52 features en 2-10 composantes tout en conservant la variance.

In [ ]:
num_clean = num_df.fillna(num_df.median())
X_scaled = StandardScaler().fit_transform(
    num_clean.drop(columns=['Churn'], errors='ignore')
)

pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

plot_pca_variance(pca_full.explained_variance_ratio_)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n80 = int(np.searchsorted(cumvar, 0.80)) + 1
n95 = int(np.searchsorted(cumvar, 0.95)) + 1
print(f'Composantes pour 80% de variance : {n80}')
print(f'Composantes pour 95% de variance : {n95}')

In [ ]:
# Projection 2D coloree par Churn
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)
labels = num_clean['Churn'].values if 'Churn' in num_clean.columns else None
plot_pca_2d(X_pca2, labels=labels, title='Projection ACP 2D - colore par Churn')

## 8. Feature Engineering

In [ ]:
df_fe = df.copy()

# 1. Ratio depenses / recency
df_fe['MonetaryPerDay'] = df_fe['MonetaryTotal'] / (df_fe['Recency'] + 1)

# 2. Panier moyen recalcule
df_fe['AvgBasketValue'] = df_fe['MonetaryTotal'] / df_fe['Frequency'].replace(0, 1)

# 3. Anciennete vs activite recente
df_fe['TenureRatio'] = df_fe['Recency'] / (df_fe['CustomerTenure'] + 1)

# 4. Frequence relative
df_fe['FrequencyRatio'] = df_fe['Frequency'] / (df_fe['CustomerTenure'] + 1)

# 5. Extraction depuis RegistDate
df_fe['RegistDate'] = pd.to_datetime(df_fe['RegistDate'], dayfirst=True, errors='coerce')
df_fe['RegYear'] = df_fe['RegistDate'].dt.year
df_fe['RegMonth'] = df_fe['RegistDate'].dt.month

print('Nouvelles features creees :')
print(df_fe[['MonetaryPerDay', 'AvgBasketValue', 'TenureRatio', 'FrequencyRatio', 'RegYear', 'RegMonth']].describe().round(2))

## 9. Lancer le Pipeline de Preprocessing Complet

In [ ]:
import subprocess, sys
res = subprocess.run(
    [sys.executable, '../src/preprocessing.py'],
    capture_output=True, text=True, cwd='..'
)
print(res.stdout)
if res.returncode != 0:
    print('ERREUR:', res.stderr)

In [ ]:
# Verifier les donnees produites
df_proc = pd.read_csv('../data/processed/retail_processed.csv')
X_train = pd.read_csv('../data/train_test/X_train.csv')
X_test = pd.read_csv('../data/train_test/X_test.csv')
y_train = pd.read_csv('../data/train_test/y_train.csv').squeeze()
y_test = pd.read_csv('../data/train_test/y_test.csv').squeeze()

print(f'Shape apres preprocessing : {df_proc.shape}')
print(f'X_train : {X_train.shape} | X_test : {X_test.shape}')
y_mean_train = y_train.mean()
y_mean_test = y_test.mean()
print(f'Churn train : {y_mean_train:.1%} | Churn test : {y_mean_test:.1%}')

## Resume des Traitements

| Etape | Action | Resultat |
|---|---|---|
| Valeurs manquantes | Imputation Age (mediane) | 267 NaN corriges |
| Outliers | Correction SupportTickets (-1, 999) | 77 corriges |
| Outliers | Correction Satisfaction (-1, 99) | 76 corriges |
| Formats | Parsing RegistDate multi-format | 4 features creees |
| Variance nulle | Suppression Newsletter | 1 feature supprimee |
| Feature brute | Engineering LastLoginIP | 2 features creees |
| Multicolinearite | Seuil |r| > 0.85 | ~5 features supprimees |
| **Total** | **52 features brutes** | **74 features finales** |
